In [6]:
import time
import sys
import os

# Add the parent directory to sys.path
notebook_path = os.getcwd()  # Get the current working directory where the notebook is located
parent_directory = os.path.dirname(notebook_path)  # Move to the parent directory
sys.path.append(parent_directory)

# Now you can import config
from config import *
from Minimal_Model_2D import *
import time
from multiprocessing import Pool

np.set_printoptions(threshold=sys.maxsize)

In [7]:
class Minimal_Model_Subband_Basis(twoD_Minimal_Model):

    def Ham(self, dis_mag=0, uncorr_var=1, lmbda=0, seed=None, kx=0, ky=0, phi=0):

        M = bmat([
           [self.H_same_spin(dis_mag, uncorr_var, lmbda, seed, kx, ky ), self.Del_array(phi) ],
           [self.Del_array(phi).conjugate(),-self.H_same_spin(dis_mag, uncorr_var, lmbda, seed, -kx, -ky).conjugate() ]
       ]) 

        return M
    
    def U_dag_ph(self, my=1):

        M = bmat([
            [ self.U_dag(my), None],
            [None, -self.U_dag(my).conjugate()],
        ])

        return M
    
    def Ham_changed_basis(self,  dis_mag=0, uncorr_var=1, lmbda=0, seed=None, kx=0, ky=0, phi=0, my=1):

        S_dag = self.U_dag_ph(my)

        S = self.U_dag_ph(my).transpose().conjugate()

        H = self.Ham(dis_mag, uncorr_var, lmbda, seed, kx, ky,phi)

        return S_dag.dot(H).dot(S)
    
    def Compute_eigenvals(self, dis_mag=0, uncorr_var=1, lmbda=0, kx=0, ky=0, phi=0, knum=4):

        SEED = random.randint(0, 2**32 - 1)  ## seed for the disorder 

        H = self.Ham(dis_mag, uncorr_var, lmbda, seed=SEED, kx=kx, ky=ky, phi=phi)
        E= eigsh(H ,k=knum ,sigma=0, return_eigenvectors=False)

        Esorted = np.sort(E)/self.delta

        return Esorted

        

In [9]:
NSM = 60            # number of sm x lattice points
M = 60              # should be atleast thrice confinement length
AX = 5              # lattice spacing in nm
AY = 5              # lattice spacing in nm
DELTA = 0.165       # in meV
MEFF_SC = 0.023     # proximitized InAs
MEFF_SM = 0.023     # InAs
NSC = 800
NTOT = 2*NSC + NSM
PHI = 0.5*np.pi 
#ALPHAT = 400        # meV-nm^3 # 20 is within perturbative regime for widths atleast up to 100 nm
K_NUM = 4           # No of eigenvalues needed to be found 
#W_MIN = 50

LY = 100           # Width in nm
OMEGA = 2*4*hbm/LY**2/MEFF_SC


print(OMEGA/2, 3*OMEGA/2)

MU_SC =   OMEGA/2 +OMEGA/10  # in between two subbands - such that we get two abs prferably 
MU_SM =   OMEGA/2 +OMEGA/10  # in between two subbands - such that we get two abs prferably

print(MU_SM)

UNCORR_VAR = 1167  ### uncorrelated variance of disorder

NUM_WORKERS = 16

DIS_MAG=0

KX=0

KY=0

NUM_ITR = 20

LMBDA = 0.15

A = Minimal_Model_Subband_Basis(NTOT,NSM,NSC,M,AX,AY,OMEGA, delta=DELTA, meff_sc=MEFF_SC, meff_sm=MEFF_SM, mu_sm = MU_SM, mu_sc=MU_SC)

t1 = time.time()

Eavg = np.empty((K_NUM))

with Pool(processes=NUM_WORKERS) as pool:

    print('started')
        
    results = pool.starmap(A.Compute_eigenvals, [(DIS_MAG, UNCORR_VAR, LMBDA, KX, KY, PHI, K_NUM) for i in range(NUM_ITR)])
    
Eavg = np.sum(results, axis=0) / NUM_ITR

t2 = time.time()

print(t2-t1)

print(Eavg)

0.6631720133634798 1.9895160400904393
0.7958064160361757
started


: 

In [3]:
### Averaging over multiple disorder samples ###

NSM = 60            # number of sm x lattice points
M = 60              # should be atleast thrice confinement length
AX = 5              # lattice spacing in nm
AY = 5              # lattice spacing in nm
DELTA = 0.165       # in meV
MEFF_SC = 0.023     # proximitized InAs
MEFF_SM = 0.023     # InAs
NSC = 800
NTOT = 2*NSC + NSM
PHI = 0.5*np.pi 
#ALPHAT = 400        # meV-nm^3 # 20 is within perturbative regime for widths atleast up to 100 nm
K_NUM = 4           # No of eigenvalues needed to be found 
#W_MIN = 50

LY = 100           # Width in nm
OMEGA = 2*4*hbm/LY**2/MEFF_SC


print(OMEGA/2, 3*OMEGA/2)

MU_SC =   OMEGA/2 +OMEGA/10  # in between two subbands - such that we get two abs prferably 
MU_SM =   OMEGA/2 +OMEGA/10  # in between two subbands - such that we get two abs prferably

print(MU_SM)

UNCORR_VAR = 1167  ### uncorrelated variance of disorder

#LMBDA = np.array([0, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1,1])
#LMBDA = np.linspace(0.075, 0.2, 51)
#LMBDA = np.linspace(0.15, 0.21, 81)
LMBDA= 0.15

Eavg = np.zeros((K_NUM))

for i in range(20):

    SEED = random.randint(0, 2**32 - 1)  ## seed for the disorder 

    A = Minimal_Model_Subband_Basis(NTOT,NSM,NSC,M,AX,AY,OMEGA, delta=DELTA, meff_sc=MEFF_SC, meff_sm=MEFF_SM, mu_sm = MU_SM, mu_sc=MU_SC)
    H = A.Ham(uncorr_var= UNCORR_VAR, lmbda= LMBDA, seed= SEED, phi=PHI)
    E= eigsh(H ,k=K_NUM ,sigma=0, return_eigenvectors=False)

    #sorted_index = np.argsort(E)

    Esorted = np.sort(E)/DELTA
    #Vsorted = V[:,sorted_index]

    Eavg += Esorted 

    if i%5 == 0 :
        print(i)

    
print(Eavg/50)

0.6631720133634798 1.9895160400904393
0.7958064160361757
0
5
10
15
[-0.38799598 -0.19375898  0.19375898  0.38799598]


In [ ]:
#LMBDA = np.linspace(0.075, 0.2, 51)

NSM = 60            # number of sm x lattice points
M = 60              # should be atleast thrice confinement length
AX = 5              # lattice spacing in nm
AY = 5              # lattice spacing in nm
DELTA = 0.165       # in meV
MEFF_SC = 0.023     # proximitized InAs
MEFF_SM = 0.023     # InAs
NSC = 800
NTOT = 2*NSC + NSM
PHI = 0.5*np.pi 
#ALPHAT = 400        # meV-nm^3 # 20 is within perturbative regime for widths atleast up to 100 nm
K_NUM = 8           # No of eigenvalues needed to be found 
#W_MIN = 50

LY = 100           # Width in nm
OMEGA = 2*4*hbm/LY**2/MEFF_SC

print(OMEGA/2, 3*OMEGA/2)

MU_SC =   OMEGA/2 +OMEGA/10  # in between two subbands - such that we get two abs prferably 
MU_SM =   OMEGA/2 +OMEGA/10  # in between two subbands - such that we get two abs prferably

print(MU_SM)


UNCORR_VAR = 1167  ### uncorrelated variance of disorder

#LMBDA = np.array([0, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1,1])
LMBDA = np.linspace(0.0, 0.25, 21)
#LMBDA = np.linspace(0.15, 0.21, 81)
#LMBDA=0
SEED = 42  ## seed for the disorder 

Ep = np.empty((LMBDA.shape[0],K_NUM))

for i in range(LMBDA.shape[0]):

    t1 = time.time()

    
    A = Minimal_Model_Subband_Basis(NTOT,NSM,NSC,M,AX,AY,OMEGA, delta=DELTA, meff_sc=MEFF_SC, meff_sm=MEFF_SM, mu_sm = MU_SM, mu_sc=MU_SC)
    H = A.Ham(uncorr_var=UNCORR_VAR, lmbda=LMBDA[i], seed=SEED, phi=PHI)
    E = eigsh(H ,k=K_NUM ,sigma=0, return_eigenvectors=False)

    Esorted = np.sort(E)

    Ep[i,:] = Esorted/DELTA

    t2 = time.time()

    print("sample no. {}".format(i), "time: ", round(t2-t1), "lmbda: ", LMBDA[i])

    #print(Ep[i,:])


np.save('Ep_no_soc_one_spin_vs_lmbda_{:.2f}_to_{:.2f}_mu_{:.2f}.npy'.format(LMBDA[0],LMBDA[-1],MU_SC),Ep)